<a href="https://colab.research.google.com/github/bachaudhry/my-llm-from-scratch/blob/main/supplementary/pre-training/gpt-to-llama/converting_llama2_to_llama3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Converting Llama 2 to Llama 3

- In this notebook, we will code along with Sebastain's source material which converts Llama 2 to Llama 3 and Llama 3.1 in a step by step fashion.
- In depth details on both models can be found in the following papers:
 - [Llama 2: Open Foundation and Fine-Tuned Chat Models (2023)](https://arxiv.org/abs/2307.09288)
 - [The Llama 3 Herd of Models](https://arxiv.org/abs/2407.21783)

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/gpt-to-llama/gpt2-to-llama2-llama3.webp?1">

In [1]:
from importlib.metadata import version
pkgs = [
    "blobfile",         # to download pretrained weights
    "huggingface_hub",
    "matplotlib",
    "tiktoken",
    "torch",
]
for p in pkgs:
    print(f"{p} version: {version(p)}")

blobfile version: 3.3.0
huggingface_hub version: 1.33.0
matplotlib version: 3.10.0
tiktoken version: 0.14.0
torch version: 2.11.0+cpu


# **1. Llama Model Conversion**

- The notebook [converting_gpt_to_llama2](https://github.com/bachaudhry/my-llm-from-scratch/blob/main/supplementary/pre-training/gpt-to-llama/converting_gpt_to_llama2.ipynb), recreated the implementation of Llama 2's components i.e. RMSNorm layers, SilU and SwiGLU activations, RoPE and SentencePiece tokenizer.
- Moving to the conversion of Llama 2 to Llama 3, we will specifically look at:
    - Modifications to the rotary embeddings
    - Implementing GQA.
    - Using a customized version of the GPT-4 tokenizer.

## **1.1 Reusing Llama 2 Components**

- Both Llama 2 and Llama 3 have lots of similarities so most of the building blocks can be imported from the notebook which dealt with the conversion of GPT 2 to Llama 2.

In [2]:
!git clone https://github.com/bachaudhry/my-llm-from-scratch.git

Cloning into 'my-llm-from-scratch'...
remote: Enumerating objects: 926, done.
remote: Counting objects: 100% (317/317), done.
remote: Compressing objects: 100% (280/280), done.
remote: Total 926 (delta 87), reused 70 (delta 24), pack-reused 609 (from 1)
Receiving objects: 100% (926/926), 2.10 MiB | 29.86 MiB/s, done.
Resolving deltas: 100% (468/468), done.


In [3]:
import os, sys, io, nbformat, types, ast

repo_folder = 'my-llm-from-scratch/supplementary/pre-training/gpt-to-llama'
os.chdir(repo_folder)
sys.path.append('.') # Add current directory of the repo to the path


In [6]:
import os
import sys
import io
import nbformat
import types
import ast
import torch
import torch.nn as nn

def import_from_notebook():
    """
    Dynamically imports specific function and class definitions from a Jupyter Notebook.
    It uses AST parsing to safely extract targeted definitions without executing
    unwanted code from the notebook's cells, and injects 'torch' and 'torch.nn as nn'
    into the execution namespace.
    """
    def import_definitions_from_notebook(fullname, names):
        current_dir = os.getcwd()
        path = os.path.join(current_dir, fullname + ".ipynb")
        path = os.path.normpath(path)

        # Load the notebook
        if not os.path.exists(path):
            raise FileNotFoundError(f"Notebook file not found at: {path}")

        with io.open(path, "r", encoding="utf-8") as f:
            nb = nbformat.read(f, as_version=4)

        # Create a module to store the imported functions and classes
        mod = types.ModuleType(fullname)
        sys.modules[fullname] = mod
        mod_dict = mod.__dict__

        # Inject necessary dependencies that were skipped by the AST parser
        mod_dict['torch'] = torch
        mod_dict['nn'] = nn

        # Go through the notebook cells and safely extract definitions using AST
        for cell in nb.cells:
            if cell.cell_type == "code":
                cell_code = cell.source
                try:
                    # Parse the cell code into an Abstract Syntax Tree
                    tree = ast.parse(cell_code)
                    for node in tree.body:
                        # Check if the node is a function or class definition
                        if isinstance(node, (ast.FunctionDef, ast.ClassDef)):
                            if node.name in names:
                                # Wrap the single definition in a module to compile it
                                module_ast = ast.Module(body=[node], type_ignores=[])
                                code_obj = compile(module_ast, filename=f"<notebook_{fullname}>", mode="exec")
                                # Execute ONLY the definition code
                                exec(code_obj, mod_dict)
                except SyntaxError:
                    # Skip cells with syntax errors
                    continue
        return mod

    fullname = "converting_gpt_to_llama2"
    names = ["precompute_rope_params", "compute_rope", "SiLU", "FeedForward", "RMSNorm", "MultiHeadAttention"]

    return import_definitions_from_notebook(fullname, names)


In [7]:
imported_module = import_from_notebook()

# We need to redefine precompute_rope_params
# precompute_rope_params = getattr(imported_module, "precompute_rope_params", None)
compute_rope = getattr(imported_module, "compute_rope", None)
SiLU = getattr(imported_module, "SiLU", None)
FeedForward = getattr(imported_module, "FeedForward", None)
RMSNorm = getattr(imported_module, "RMSNorm", None)

# MultiHeadAttention only for comparison purposes
MultiHeadAttention = getattr(imported_module, "MultiHeadAttention", None)

In [12]:
# Print the non-dunder attributes of the imported module to verify extraction
print([attr for attr in dir(imported_module) if not attr.startswith('__')])

['FeedForward', 'MultiHeadAttention', 'RMSNorm', 'SiLU', 'compute_rope', 'nn', 'precompute_rope_params', 'torch']
